# General Requirements 

In [4]:
%pip install beautifulsoup4
%pip install lxml
%pip install spacy
%pip install ipywidgets
%pip install transformers

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
     |████████████████████████████████| 139 kB 2.6 MB/s eta 0:00:01
     |████████████████████████████████| 214 kB 17.6 MB/s eta 0:00:01
     |████████████████████████████████| 2.3 MB 54.3 MB/s eta 0:00:01
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


# Prepare the dataset

### Open the dataset

In [14]:
from bs4 import BeautifulSoup


# Reading the data inside the xml
# file to a variable under the name
# data
with open('deid_surrogate_train_all_version2.xml', 'r') as f:
    data = f.read()

# Passing the stored data inside
# the beautifulsoup parser, storing
# the returned object
Bs_data = BeautifulSoup(data, "xml")

# Using find() to extract attributes
# of the first instance of the tag
b_type = Bs_data.find_all('PHI', {'TYPE':'HOSPITAL'})

print(b_type)

[<PHI TYPE="HOSPITAL">FIH</PHI>, <PHI TYPE="HOSPITAL">Sephsandpot Center</PHI>, <PHI TYPE="HOSPITAL">Valtawnprinceel Community Memorial Hospital</PHI>, <PHI TYPE="HOSPITAL">Valtawnprinceel
            Community Memorial Hospital</PHI>, <PHI TYPE="HOSPITAL">Em Nysonken Medical Center</PHI>, <PHI TYPE="HOSPITAL">OLH</PHI>, <PHI TYPE="HOSPITAL">HLGMC</PHI>, <PHI TYPE="HOSPITAL">Staviewordna University Of Medical Center</PHI>, <PHI TYPE="HOSPITAL">CMC</PHI>, <PHI TYPE="HOSPITAL">CMC</PHI>, <PHI TYPE="HOSPITAL">CMC</PHI>, <PHI TYPE="HOSPITAL">6U-489</PHI>, <PHI TYPE="HOSPITAL">HLGMC</PHI>, <PHI TYPE="HOSPITAL">Hoseocon Medical Center</PHI>, <PHI TYPE="HOSPITAL">Heaonboburg Linpack Grant Medical Center</PHI>, <PHI TYPE="HOSPITAL">HLGMC</PHI>, <PHI TYPE="HOSPITAL">Liccam Community Medical
            Center</PHI>, <PHI TYPE="HOSPITAL">Liccam Community Medical Center</PHI>, <PHI TYPE="HOSPITAL">HLGMC</PHI>, <PHI TYPE="HOSPITAL">CMC</PHI>, <PHI TYPE="HOSPITAL">1D-419</PHI>, <PHI TYPE="HOSPITAL"

In [15]:
from spacy import displacy
import re

xml_text = Bs_data.get_text()
record_test = Bs_data.find('RECORD')

### Remove IDs if needed

In [ ]:
def remove_ids(soup):
    for item in soup.find_all(attrs={"TYPE": "ID"}):
        item.decompose()
    return soup

In [41]:
record_test = remove_ids(record_test) # Remove IDs from the XML data
record_text = record_test.get_text()

# Spacy

In [42]:
nlp.max_length = len(record_str) + 100  # add a bit of a buffer
doc_xml = nlp(record_str)

#displacy.serve(doc_xml, style="ent")

In [43]:
# Define the entities to mask
entities_to_mask = ["PERSON", "EMAIL", "GPE", "DATE", "LOC", "FAC"]
pattern_date = re.compile("[0-9]{2}\/[0-9]{2}\/[0-9]{2,4}")

# Function to mask entities
def mask_entities(doc, entities_to_mask):
    masked_text = doc.text
    for ent in doc.ents:
        if ent.label_ in entities_to_mask:
            masked_text = masked_text.replace(ent.text, "DATE")
        if pattern_date.match(ent.text):
            masked_text = masked_text.replace(ent.text, "DATE")
    return masked_text

In [44]:
# Mask the entities
masked_text = mask_entities(doc_xml, entities_to_mask)

# Print the masked text
print(masked_text)

masked_xml = nlp(masked_text)
#displacy.serve(masked_xml, style="ent")

<RECORD ID="640">
<TEXT>
<PHI TYPE="ID">*****</PHI>
<PHI TYPE="HOSPITAL">FIH</PHI>
<PHI TYPE="ID">*****</PHI>
<PHI TYPE="ID">*****</PHI>
<PHI TYPE="ID">*****</PHI>
<PHI TYPE="DATE">11/19</PHI>/1994 12:00:00 AM
Discharge Summary
Unsigned
DIS
Report Status :
Unsigned
ADMISSION DATE :
<PHI TYPE="DATE">11/19</PHI>/94
DISCHARGE DATE :
<PHI TYPE="DATE">11/DATE</PHI>/94
ADMISSION DIAGNOSIS :
Aspiration pneumonia , esophageal laceration .
HIDATEORY OF PRESENT ILLNESS :
Mr. <PHI TYPE="PATIENT">Blind</PHI> is a DATE white white male with a history of diabetes mellitus , inferior myocardial infarction , who underwent open repair of his increased diverticulum <PHI TYPE="DATE">November 13th</PHI> at <PHI TYPE="HOSPITAL">Sephsandpot Center</PHI> .
The patient developed hematemesis <PHI TYPE="DATE">November 15th</PHI> and was intubated for respiratory distress .
He was transferred to the <PHI TYPE="HOSPITAL">Valtawnprinceel Community Memorial Hospital</PHI> for endoscopy and esophagoscopy on the <PHI

# Bio NER

### Load the specialized model

In [3]:
# Use a pipeline as a high-level helper
from transformers import pipeline

pipe = pipeline("token-classification", model="alvaroalon2/biobert_diseases_ner")

/home/cbrice/.local/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Device set to use cpu


# Finetuning BERT

In [5]:
%pip install datasets evaluate transformers[sentencepiece]
%pip install accelerate
# To run the training on TPU, you will need to uncomment the following line:
# !pip install cloud-tpu-client==0.10 torch==1.9.0 https://storage.googleapis.com/tpu-pytorch/wheels/torch_xla-1.9-cp37-cp37m-linux_x86_64.whl
# !apt install git-lfs

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [6]:
from bs4 import BeautifulSoup
from datasets import Dataset, DatasetDict
from transformers import AutoTokenizer, AutoModelForMaskedLM, DataCollatorForLanguageModeling, Trainer, TrainingArguments
import torch
import collections
import numpy as np
from transformers import default_data_collator
import math

In [ ]:
# Step 1: Parse the XML file with BeautifulSoup
def parse_xml(file_path):
    with open(file_path, 'r') as f:
        data = f.read()

    bs_data = BeautifulSoup(data, "lxml-xml")  # Use lxml-xml parser

    # Print the structure of the XML to verify tag names
    # print(bs_data.prettify())

    parsed_data = []
    for item in bs_data.find_all('RECORD'):  # Adjust the tag name as per your XML structure
        text = item.find('TEXT').text
        label = item.find("SMOKING",).get("STATUS")
        parsed_data.append({'text': text, 'label': label})

    return parsed_data

In [ ]:
# Step 2: Convert to Dictionary Format
def convert_to_dict_format(data):
    dict_format = {'text': [], 'label': []}
    for entry in data:
        dict_format['text'].append(entry['text'])
        dict_format['label'].append(entry['label'])
    return dict_format

In [ ]:
# Step 3: Create a dataset from the parsed data
def create_dataset(data):
    return Dataset.from_dict(data)

# File path to your XML file
file_path = 'smokers_surrogate_train_all_version2.xml'

# Parse the XML file
parsed_data = parse_xml(file_path)
print(f"Parsed data size: {len(parsed_data)}")  # Debugging statement

# Convert to dictionary format
dict_format_data = convert_to_dict_format(parsed_data)
print(f"Dictionary format data size: {len(dict_format_data['text'])}")  # Debugging statement

# Create a dataset
dataset = create_dataset(dict_format_data)

# Check the dataset size
print(f"Dataset size: {len(dataset)}")

# Shuffle and select samples
dataset_size = len(dataset)
if dataset_size > 0:
    sample = dataset.shuffle(seed=42).select(range(min(3, dataset_size)))

    # Print the samples
    for row in sample:
        # print(f"\n'>>> Review: {row['text']}'")
        print(f"'>>> Label: {row['label']}'")
else:
    print("Dataset is empty.")

In [ ]:
# Split the dataset into train and test
if dataset_size > 0:
    train_test_split = dataset.train_test_split(test_size=0.1)
    train_dataset = train_test_split['train']
    test_dataset = train_test_split['test']

    # Step 4: Tokenize the dataset
    model_checkpoint = "distilbert-base-uncased"
    tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

    def tokenize_function(examples):
        result = tokenizer(examples["text"], padding=True, truncation=True)
        # if tokenizer.is_fast:
        #     result["word_ids"] = [result.word_ids(i) for i in range(len(result["input_ids"]))]
        return result

    # Use batched=True to activate fast multithreading!
    tokenized_datasets = DatasetDict({
        'train': train_dataset.map(tokenize_function, batched=True, remove_columns=["text", "label"]),
        'test': test_dataset.map(tokenize_function, batched=True, remove_columns=["text", "label"])
    })

    # Step 5: Group texts into chunks
    chunk_size = 128

    def group_texts(examples):
        # Concatenate all texts
        concatenated_examples = {k: sum(examples[k], []) for k in examples.keys()}
        # Compute length of concatenated texts
        total_length = len(concatenated_examples[list(examples.keys())[0]])
        # We drop the last chunk if it's smaller than chunk_size
        total_length = (total_length // chunk_size) * chunk_size
        # Split by chunks of max_len
        result = {
            k: [t[i : i + chunk_size] for i in range(0, total_length, chunk_size)]
            for k, t in concatenated_examples.items()
        }
        # Create a new labels column
        result["labels"] = result["input_ids"].copy()
        return result

    lm_datasets = tokenized_datasets.map(group_texts, batched=True)

    # Step 6: Fine-tune the model
    data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm_probability=0.15)

    model = AutoModelForMaskedLM.from_pretrained(model_checkpoint)

    training_args = TrainingArguments(
        output_dir="./results",
        evaluation_strategy="epoch",
        learning_rate=2e-5,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        num_train_epochs=3,
        weight_decay=0.01,
        remove_unused_columns=False,  # Ensure this is set to False
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=lm_datasets["train"],
        eval_dataset=lm_datasets["test"],
        data_collator=data_collator,
    )
    eval_results = trainer.evaluate()
    print(f">>> Perplexity: {math.exp(eval_results['eval_loss']):.2f}")
    trainer.train()
    # Evaluate the model
    eval_results = trainer.evaluate()
    print(f">>> Perplexity: {math.exp(eval_results['eval_loss']):.2f}")
else:
    print("Dataset is empty. Cannot proceed with training.")
